# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hussaintinwala2/Flyrank/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

*The research question and the decision it supports.*

## 1. Question

### Research question

Can decision-point content signals be used to prioritize pages for human review when there are indications of low search-result click-through or content staleness?

This work evaluates whether a simple learned model can provide useful directional signal beyond a transparent rule-based baseline, while using a validation design that prevents pages from the same client appearing in both training and test sets.

### Decision supported

The intended decision is which content pages should be reviewed first and what type of review may be appropriate.

The output is decision-support for human reviewers. It is not intended to automatically rewrite, publish, delete, redirect, or otherwise modify content.

## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

### Dataset and scope

This analysis uses the FlyRank ML Internship dataset and focuses on content-level search-performance signals available at the decision point.

The analysis uses hashed client and content identifiers so that the public-facing research artifact does not expose client names, URLs, or private queries.

### Decision-point window

The predictive features are taken from March 2026:

- March impressions
- March clicks
- March average search position

These features represent information available before the April 2026 outcome window.

### Outcome window

The target is `is_declining`, defined using the April 2026 outcome window. The target therefore represents a subsequent outcome rather than an input to the predictive model.

### Exclusions

Future outcome fields were excluded from the predictive feature set to avoid target leakage.

`client_hash_id` was not used as a predictive feature. It was retained only for grouped validation so that pages from the same client could not appear in both the training and test sets.

Client names, URLs, private queries, and other identifying information were not included in the public-facing analysis.

### Modeling population

The final modeling dataset contained 176,737 usable rows across 47 clients, with a positive target rate of approximately 53.2%.

The analysis is therefore a decision-support study on the available dataset rather than a claim about all web content or all search environments.

In [6]:
# Reconnect to the FlyRank warehouse

import os
import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

REL = "hf://datasets/FlyRank/internship-warehouse"

TABLES = {
    "dim_clients": f"read_parquet('{REL}/dim_clients.parquet')",
    "dim_content": f"read_parquet('{REL}/dim_content.parquet')",
    "fact_daily": f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    "fact_query_90d": f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

print("DuckDB connected successfully.")

DuckDB connected successfully.


In [7]:
model_data = con.sql(f"""
    SELECT
        f.client_hash_id,
        f.content_hash_id,
        f.report_date,
        f.gsc_impressions,
        f.gsc_clicks,
        f.gsc_avg_position,
        c.content_created_date,
        c.content_updated_date
    FROM {TABLES['fact_daily']} f
    LEFT JOIN {TABLES['dim_content']} c
        ON f.client_hash_id = c.client_hash_id
        AND f.content_hash_id = c.content_hash_id
    WHERE f.report_date >= DATE '2026-03-01'
      AND f.report_date < DATE '2026-04-01'
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [18]:
# Restore the exact FlyRank warehouse table mapping used in W05

REL = "hf://datasets/FlyRank/internship-warehouse"

TABLES = {
    "dim_clients": f"read_parquet('{REL}/dim_clients.parquet')",
    "dim_content": f"read_parquet('{REL}/dim_content.parquet')",
    "fact_daily": f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    "fact_query_90d": f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

In [15]:
import duckdb
from google.colab import userdata

# Get the Hugging Face token from Colab Secrets
HF_TOKEN = userdata.get("HF_TOKEN")

if not HF_TOKEN:
    raise ValueError(
        "HF_TOKEN was not found. Add your Hugging Face token "
        "to Colab Secrets with the name HF_TOKEN."
    )

print("HF_TOKEN loaded:", bool(HF_TOKEN))

# Fresh DuckDB connection
con = duckdb.connect()

# Authenticate DuckDB with Hugging Face
con.execute(
    f"""
    CREATE OR REPLACE SECRET hf (
        TYPE huggingface,
        TOKEN '{HF_TOKEN}'
    )
    """
)

print("DuckDB Hugging Face authentication configured.")
REL = "hf://datasets/FlyRank/internship-warehouse"

TABLES = {
    "dim_clients": f"read_parquet('{REL}/dim_clients.parquet')",
    "dim_content": f"read_parquet('{REL}/dim_content.parquet')",
    "fact_daily": f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    "fact_query_90d": f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

print(TABLES)
test = con.sql(f"""
    SELECT *
    FROM {TABLES['dim_content']}
    LIMIT 5
""").df()

display(test)

HF_TOKEN loaded: True
DuckDB Hugging Face authentication configured.
{'dim_clients': "read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_clients.parquet')", 'dim_content': "read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_content.parquet')", 'fact_daily': "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet')", 'fact_query_90d': "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_query_90d.parquet')"}


,client_hash_id,content_hash_id,keyword_hash_id,url_hash_id,keyword_char_count,keyword_token_count,url_char_count,content_created_date,content_updated_date,content_type,...,category_count,keyword_created_date,provider_used,model_used,char_count,word_count,last_optimized_date,optimization_eligible_date,is_published,is_deleted
0,client_04660893ae39614a,content_004de9653278b5a4,keyword_e754999ab88dd9f2,url_d6091f18cf628794,22,4,108,2026-05-30,2026-07-01,keyword article,...,3,2026-05-12,gemini-generate-content,gemini-3-flash-preview,15682,2555,NaT,NaT,True,False
1,client_04660893ae39614a,content_00dc5efae381b2ab,keyword_4329d7aede8e208b,url_3a66d2f2e36823ca,31,6,95,2026-06-12,2026-07-01,keyword article,...,4,2026-06-01,gemini-generate-content,gemini-3-flash-preview,15438,2430,NaT,NaT,True,False
2,client_04660893ae39614a,content_01410f2556c327ac,keyword_9b08047d3d2a0406,url_809eda7a7e20b3b2,22,5,82,2026-05-09,2026-07-01,keyword article,...,4,2026-05-06,gemini-generate-content,gemini-3-flash-preview,16576,2645,NaT,NaT,True,False
3,client_04660893ae39614a,content_019f27f634053ca7,keyword_e7cec7ab1804c1c2,url_5fb42bafc4399861,14,3,92,2026-06-15,2026-06-15,keyword article,...,4,2026-06-01,gemini-generate-content,gemini-3-flash-preview,15457,2522,NaT,NaT,True,False
4,client_04660893ae39614a,content_01efa71faea45dcc,keyword_56b0062a1d8b7524,url_ece0abc3e5fb75f9,24,6,98,2026-05-21,2026-06-01,keyword article,...,4,2026-05-12,gemini-generate-content,gemini-3-flash-preview,15776,2552,NaT,NaT,True,False


In [16]:
import numpy as np
import pandas as pd

# Prepare the W05 modeling dataset
model_data = con.sql(f"""
    SELECT
        f.client_hash_id,
        f.content_hash_id,
        f.report_date,
        f.gsc_impressions,
        f.gsc_clicks,
        f.gsc_avg_position,
        c.content_created_date,
        c.content_updated_date
    FROM {TABLES['fact_daily']} f
    LEFT JOIN {TABLES['dim_content']} c
        ON f.client_hash_id = c.client_hash_id
        AND f.content_hash_id = c.content_hash_id
    WHERE f.report_date >= DATE '2026-03-01'
      AND f.report_date < DATE '2026-04-01'
""").df()

# Ensure date columns are datetime
model_data["report_date"] = pd.to_datetime(model_data["report_date"])
model_data["content_created_date"] = pd.to_datetime(
    model_data["content_created_date"]
)
model_data["content_updated_date"] = pd.to_datetime(
    model_data["content_updated_date"]
)

# Decision-point features
model_data["days_since_update"] = (
    model_data["report_date"] -
    model_data["content_updated_date"]
).dt.days

model_data["content_age_days"] = (
    model_data["report_date"] -
    model_data["content_created_date"]
).dt.days

model_data["ctr"] = (
    model_data["gsc_clicks"] /
    model_data["gsc_impressions"].replace(0, np.nan)
)

feature_cols = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "days_since_update",
    "content_age_days",
    "ctr"
]

# Keep only usable modeling rows
model_data = model_data.dropna(
    subset=feature_cols + ["client_hash_id"]
).copy()

print(f"Modeling rows: {len(model_data):,}")
print(f"Clients: {model_data['client_hash_id'].nunique():,}")
print("\nFeatures:")
print(feature_cols)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Modeling rows: 3,611,061
Clients: 47

Features:
['gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'days_since_update', 'content_age_days', 'ctr']


In [17]:
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# W05 decision-point dataset
# One row per content page using March 2026 aggregates
# ------------------------------------------------------------

model_data = con.sql(f"""
WITH march_metrics AS (
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(gsc_impressions) AS impressions_march,
        SUM(gsc_clicks) AS clicks_march,
        AVG(gsc_avg_position) AS avg_position_march

    FROM {TABLES['fact_daily']}

    WHERE report_date >= DATE '2026-03-01'
      AND report_date < DATE '2026-04-01'

    GROUP BY
        client_hash_id,
        content_hash_id
)

SELECT
    m.client_hash_id,
    m.content_hash_id,

    m.impressions_march,
    m.clicks_march,
    m.avg_position_march,

    c.content_created_date,
    c.content_updated_date

FROM march_metrics m

LEFT JOIN {TABLES['dim_content']} c
    ON m.client_hash_id = c.client_hash_id
    AND m.content_hash_id = c.content_hash_id
""").df()

# ------------------------------------------------------------
# Date handling
# ------------------------------------------------------------

model_data["content_created_date"] = pd.to_datetime(
    model_data["content_created_date"],
    errors="coerce"
)

model_data["content_updated_date"] = pd.to_datetime(
    model_data["content_updated_date"],
    errors="coerce"
)

# March 31 is the decision point
DECISION_DATE = pd.Timestamp("2026-03-31")

model_data["days_since_update"] = (
    DECISION_DATE -
    model_data["content_updated_date"]
).dt.days

model_data["content_age_days"] = (
    DECISION_DATE -
    model_data["content_created_date"]
).dt.days

# CTR from March observations
model_data["ctr"] = (
    model_data["clicks_march"] /
    model_data["impressions_march"].replace(0, np.nan)
)

# Same feature set used in W05
feature_cols = [
    "impressions_march",
    "clicks_march",
    "avg_position_march"
]

# Keep usable rows
model_data = model_data.dropna(
    subset=feature_cols + ["client_hash_id"]
).copy()

print("=== CAPSTONE DATASET ===")
print(f"Modeling rows: {len(model_data):,}")
print(f"Clients: {model_data['client_hash_id'].nunique():,}")
print(f"Features: {feature_cols}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

=== CAPSTONE DATASET ===
Modeling rows: 176,738
Clients: 47
Features: ['impressions_march', 'clicks_march', 'avg_position_march']


## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

The learned model is Logistic Regression because it provides an interpretable, probability-based first model and is appropriate for directional decision support. The predictive features are March 2026 impressions, clicks, and average search position. The target is_declining is defined from the subsequent April 2026 outcome, keeping the outcome after the decision point.

The Week-4 baseline provides the comparison point. Model performance is evaluated using ROC-AUC. A random split produced an observed ROC-AUC of 0.586, while the client-grouped split produced an observed ROC-AUC of 0.538 with zero client overlap between training and test sets. The grouped result is treated as the more honest estimate because it tests whether the learned signal transfers to clients not seen during training.

In [21]:
import numpy as np
import pandas as pd

from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

# ============================================================
# 1. Recreate the exact W05 target
# ============================================================

april_data = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS impressions_april
    FROM {TABLES['fact_daily']}
    WHERE report_date >= DATE '2026-04-01'
      AND report_date < DATE '2026-05-01'
    GROUP BY
        client_hash_id,
        content_hash_id
""").df()

# Avoid duplicate target columns if this cell is accidentally
# run more than once.
model_data = model_data.drop(
    columns=["impressions_april", "is_declining"],
    errors="ignore"
)

model_data = model_data.merge(
    april_data,
    on=["client_hash_id", "content_hash_id"],
    how="left"
)

model_data["impressions_april"] = (
    model_data["impressions_april"].fillna(0)
)

# EXACT W05 target:
# April impressions < 80% of March impressions
model_data["is_declining"] = (
    model_data["impressions_april"]
    < 0.80 * model_data["impressions_march"]
).astype(int)

# ============================================================
# 2. Final modeling rows
# ============================================================

model_data = model_data.dropna(
    subset=feature_cols + [
        "is_declining",
        "client_hash_id"
    ]
).copy()

X = model_data[feature_cols]
y = model_data["is_declining"]
groups = model_data["client_hash_id"]

print("=== METHODOLOGY DATA CHECK ===")
print(f"Usable modeling rows: {len(model_data):,}")
print(f"Clients: {model_data['client_hash_id'].nunique():,}")
print(f"Positive rate: {y.mean():.3f}")

print("\nTarget distribution:")
print(y.value_counts())

# ============================================================
# 3. Client-grouped validation
# ============================================================

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.25,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

train_clients = set(
    groups.iloc[train_idx]
)

test_clients = set(
    groups.iloc[test_idx]
)

print("\n=== GROUPED VALIDATION ===")
print(f"Training rows: {len(X_train):,}")
print(f"Test rows:     {len(X_test):,}")
print(f"Training clients: {len(train_clients)}")
print(f"Test clients:     {len(test_clients)}")
print(f"Client overlap:   {len(train_clients & test_clients)}")

# ============================================================
# 4. Logistic Regression
# ============================================================

model = Pipeline([
    ("scaler", StandardScaler()),
    (
        "logistic",
        LogisticRegression(
            max_iter=1000,
            random_state=42
        )
    )
])

model.fit(X_train, y_train)

test_prob = model.predict_proba(X_test)[:, 1]

roc_auc = roc_auc_score(
    y_test,
    test_prob
)

print("\n=== LOGISTIC REGRESSION ===")
print(f"ROC-AUC: {roc_auc:.3f}")

# ============================================================
# 5. Leakage / feature audit
# ============================================================

print("\n=== LEAKAGE / FEATURE CHECK ===")

print("Predictive features:")
for feature in feature_cols:
    print(f" - {feature}")

future_fields = [
    "impressions_april",
    "is_declining"
]

print("\nFuture outcome fields excluded from predictive features:")
for field in future_fields:
    print(f" - {field}")

print("\nClient identifier used as predictive feature: No")
print("Client identifier used for grouped validation: Yes")

print("\nClient overlap check:")
if len(train_clients & test_clients) == 0:
    print("PASS: no client appears in both train and test.")
else:
    print("FAIL: client overlap detected.")

print("\nMethodology interpretation:")
print(
    "The client-grouped ROC-AUC is the primary validation result "
    "because test clients are not present in training."
)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

=== METHODOLOGY DATA CHECK ===
Usable modeling rows: 176,738
Clients: 47
Positive rate: 0.532

Target distribution:
is_declining
1    94000
0    82738
Name: count, dtype: int64

=== GROUPED VALIDATION ===
Training rows: 133,474
Test rows:     43,264
Training clients: 35
Test clients:     12
Client overlap:   0

=== LOGISTIC REGRESSION ===
ROC-AUC: 0.538

=== LEAKAGE / FEATURE CHECK ===
Predictive features:
 - impressions_march
 - clicks_march
 - avg_position_march

Future outcome fields excluded from predictive features:
 - impressions_april
 - is_declining

Client identifier used as predictive feature: No
Client identifier used for grouped validation: Yes

Client overlap check:
PASS: no client appears in both train and test.

Methodology interpretation:
The client-grouped ROC-AUC is the primary validation result because test clients are not present in training.


## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

The Logistic Regression model is compared with the Week-4 baseline using the same client-grouped test split and ROC-AUC metric. This comparison is intended to show whether the learned model provides additional directional signal beyond the baseline. The grouped validation result is treated as the primary result because no client appears in both training and test sets.

Results are reported as observed and measured performance. They should be interpreted as decision-support evidence rather than proof that the model can reliably predict future content decline.

In [22]:
# ============================================================
# Section 4 — Results (vs baseline)
# ============================================================

from sklearn.metrics import roc_auc_score
import pandas as pd
import numpy as np

# ------------------------------------------------------------
# 1. Logistic Regression result
# ------------------------------------------------------------

model_prob = model.predict_proba(X_test)[:, 1]

model_auc = roc_auc_score(
    y_test,
    model_prob
)

# ------------------------------------------------------------
# 2. Week-4 baseline
# ------------------------------------------------------------
# Baseline uses the observed March search-performance signal:
# lower March impressions are treated as higher decline risk.
#
# IMPORTANT:
# The baseline is evaluated on the EXACT SAME grouped test rows
# and against the EXACT SAME April target.

baseline_score = -X_test["impressions_march"].astype(float)

baseline_auc = roc_auc_score(
    y_test,
    baseline_score
)

# ------------------------------------------------------------
# 3. Compare model and baseline
# ------------------------------------------------------------

results = pd.DataFrame({
    "method": [
        "Week-4 baseline",
        "Logistic Regression"
    ],
    "roc_auc": [
        baseline_auc,
        model_auc
    ]
})

results["difference_vs_baseline"] = (
    results["roc_auc"] - baseline_auc
)

print("=== RESULTS: MODEL VS BASELINE ===")
display(results)

print("\n=== RESULT CHECK ===")
print(f"Baseline ROC-AUC:       {baseline_auc:.3f}")
print(f"Logistic Regression:    {model_auc:.3f}")
print(f"Difference:             {model_auc - baseline_auc:+.3f}")

# ------------------------------------------------------------
# 4. Honest interpretation
# ------------------------------------------------------------

if model_auc > baseline_auc:
    print(
        "\nObserved result: Logistic Regression has a higher "
        "ROC-AUC than the baseline on the same grouped test split."
    )
elif model_auc < baseline_auc:
    print(
        "\nObserved result: Logistic Regression has a lower "
        "ROC-AUC than the baseline on the same grouped test split."
    )
else:
    print(
        "\nObserved result: Logistic Regression and the baseline "
        "have the same ROC-AUC on the grouped test split."
    )

print(
    "\nThe result is directional and decision-support oriented; "
    "it does not establish that the model will improve content performance."
)

=== RESULTS: MODEL VS BASELINE ===


,method,roc_auc,difference_vs_baseline
0,Week-4 baseline,0.521049,0.000000
1,Logistic Regression,0.537685,0.016636



=== RESULT CHECK ===
Baseline ROC-AUC:       0.521
Logistic Regression:    0.538
Difference:             +0.017

Observed result: Logistic Regression has a higher ROC-AUC than the baseline on the same grouped test split.

The result is directional and decision-support oriented; it does not establish that the model will improve content performance.


## 5. Limitations

*What this work cannot claim.*

## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [ ] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [ ] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.
